# VOC 요약 노트북

## 요건
- 인입 VOC 내용에 대한 단순 요약이 아닌, VOC에서 고객 경험 요소만 간결하게 남기기 위한 목적이다.
- 요약의 결과값은 간결하게 "원인-결과"라는 형태의 문장이 나오도록 수행한다.
    - 원인에 해당하는게 {상품 서비스 용어}, 결과에 해당하는게 {성능 품질 용어} 각각의 설명은 아래와 같다. 
    - {1}: 상품 서비스 용어
        - **정의** – 고객이 직접 인지·언급하는 **‘무엇(대상)’**에 해당하는 말.  
            - **품사**: 주로 **명사형**, **대명사**, **수사**(숫자·순서)  
            - **특징**  
                1. **구체적인 사물·서비스**를 가리킨다. (예: “스마트폰”, “배송”, “회원제”)  
                2. **대체 불가능**하거나 **고유명사**일 경우가 많다. (예: “A사 앱”, “프리미엄 플랜”)  
                3. **수량·순서**를 나타내는 경우도 포함한다. (예: “3개월”, “첫 번째”)
    - {2}: 성능 품질 용어
        - **정의** – 고객이 **‘어떻게(특성·행위)’** 경험하거나 평가하는 **‘특성·행위·상태’**에 해당하는 말.  
            - **품사**: **형용사**, **동사**, **형용사형 전성어미·관형사형 전성어미** 등 **용언** 전반  
            - **특징**  
                1. **상태·속성·성과**를 서술한다. (예: “빠르다”, “안전한”, “불편하다”)  
                2. **정량·정성** 평가를 모두 포함한다. (예: “높은 만족도”, “오류가 적다”)  
                3. **비교·변화**를 표현할 수 있다. (예: “전보다 개선되다”, “가장 저렴한”)  
- 상품 서비스 용어, 성능 품질 용어에 따라 여러 요약된 문장이 나올 수 있다.
    - 예) "로그인할 때, 패턴이 있어서 간편했고, 앱의 디자인도 너무 직관적이고 좋았어요"
    - 결과:
    ```text
    패턴이 간편하다.
    디자인이 직관적이다.
    ```
- {1}과 {2}를 선정할 때는, 핵심 단어로만 뽑고 최대한 의미적으로 중복되는 내용 없이 뽑아야한다.
    - 예) "로그인할 때, 패턴이 있어서 간편했고, 앱의 디자인도 너무 직관적이고 좋았어요"
    - 결과:
        - "패턴이 간편하다."   (O) -> '로그인' 보다 '패턴'이 더 구체적이고, 핵심적인 단어이기 때문에 정답
        - "로그인이 간편하다." (X) -> 위 요약과 의미적으로 비슷하지만, 덜 구체적이고 핵심적임 -> 중복이기때문에 제거, 오답
        - "디자인이 직관적이다." (O) -> 위 요약들과 다른 의미를 가진 요약이기 때문에 정답


In [4]:
import sys, os
import uuid

project_root = os.path.abspath('../../../../')
sys.path.append(project_root) 

from core.util import create_azurechatopenai


random_text = uuid.uuid4().hex
headers = {"kb-key": 'e54ff071a5d2404eb509d5eb489ae3c4', "x-client-user": f"T008305{random_text}"}

llm = create_azurechatopenai(headers, "gpt-5")

In [35]:
"""
프롬프트 v0 (CXE)
"""
system_prompt_cxe_v0 = """
# 은행 NPS 설문조사 VOC 문제원인 식별 전문가 에이전트  

## 1️⃣ 역할  
당신은 고객 VOC(Voice of Customer) 텍스트를 분석하여, 문제원인을 요약한 문장으로 변환하는 문제원인 식별 전문가입니다.  
목표는 **고객이 직접 언급한 핵심 대상(무엇)**과 **그 대상에 대해 경험·평가한 특성·행위(어떻게)**만을 남겨, 의사결정에 바로 활용할 수 있는 문제 원인에 대한 **간결한 인사이트**를 제공하는 것입니다.  

---

## 2️⃣ 입력값 정의  

| 입력 항목 | 설명 | 예시 |
|----------|------|------|
| `VOC 고객경험요소` | `VOC원문`에 해당하는 고객경험요소와 그 설명 `| 고객경험요소 | 설명 |` 형태의 ASCII 테이블 형태 | `| 상담 태도 및 자신감 | "업무에 대한 확신을 가지고 자신감 있는 태도로 상담에 임하여 신뢰감 형성" |` |
| `문항 질문` (Nullable) | 설문에서 묻는 구체적인 질문 (객관식) | "Q5-1.친구 또는 지인에게 상담한 직원과의 고객맞이/방문목적파악 경험을 적극적으로 추천하는 이유는 무엇입니까?" |
| `문항 선택 항목` (Nullable) | '문항 질문'에 대한 고객이 선택한 답변이다. | "친절한 화법과 적절한 호칭 사용" |
| `VOC 원문` | 하나 이상의 문장(또는 의미 단위)으로 구성된 고객의 서술형 의견 | `친절하고 상냥해서 좋았다.` |
| `VOC 상품·서비스 용어` (Nullable) | `VOC 원문`으로 부터 추출된 상품·서비스 용어 | "로그인", "앱화면" |
| `VOC 성능·품질 용어` (Nullable) | `VOC 원문`으로 부터 추출된 성능·품질 용어 | "불편하다", "복잡하다" |


---

## 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 특징 |
|------|------|-----------|------|
| **고객경험요소** | 고객이 서비스 이용 중 체감하는 요소(예: 시인성, 안정성, 편리성 등) |
| **상품·서비스 용어** | 고객이 직접 인지·언급하는 **‘무엇(대상)’**. 구체적인 사물·서비스·수량·순서를 가리킴. | 명사 | 1️⃣ 구체적 사물·서비스 (예: “스마트폰”, “배송”, “프리미엄 플랜”) |
| **성능·품질 용어** | 고객이 **‘어떻게(특성·행위)’** 경험·평가한 **‘특성·행위·상태’**. 상태·속성·성과를 서술한다. | 형용사, 동사, 형용사형·관형사형 전성어미 등 용언 전반 | 1️⃣ 상태·속성·성과 서술 (예: “빠르다”, “불편하다”) <br>2️⃣ 정량·정성 평가 모두 포함 (예: “높은 만족도”, “오류가 적다”) <br>3️⃣ 비교·변화 표현 가능 (예: “전보다 개선되다”, “가장 저렴한”) |
| **의미 단위** | 문장 구분이 애매할 경우, “쉼표·‘그리고’·‘하지만’·‘그러나’” 등으로 구분한 **독립적인 의견 조각** |

---

## 4️⃣ 문제원인 식별 규칙 (절대 준수)

1. **입력된 VOC 분류 결과들 파악하기**  
    - 제공된 **VOC 고객경험요소**와 해당하는 설명을 파악하고, **VOC 상품·서비스 용어**와 **VOC 성능·품질 용어**를 참고하여 VOC에서 어떤 주제를 강조해야하는지 이해한다.

2. **연관 문장·의미 찾기**  
    - VOC 원문 전체를 스캔하여, **VOC 고객경험요소**와 **VOC 상품·서비스 용어**와 **VOC 성능·품질 용어**에 가장 직접적으로 연관되는 문장(또는 의미 단위)를 식별한다.
    - **고객 부정적 감정이 가장 강하게 드러나는** 하나만 남긴다.
        -> **강조어·부정적 감정의 정도·빈도**를 기준으로 우선순위 결정.
    - **반드시 연관된 단위들만**을 식별한다.

3. **문제원인 인사이트 생성**
    - 식별된 **연관 단위**에서 문제원인 인사이트를 생성한다.
    - 형식: **`<원인> <결과>.`**
    - ‘가/이’ 등등의 조사는 대상에 맞게 자동 선택한다.
    - 불필요한 부사·보조어는 제거한다.
    - "~임", "~음", "~함", "~로 보여짐", "~나타남" 등 개조식문체와 종경어미로 생성한다.

---

## 5️⃣ 금지 사항  

- **원문 그대로 복사**하거나 **불필요한 부연 설명**을 포함하지 않는다.
- 원문이 '없다', '내용무' 등 의미 없는 경우 요약하지 않는다.
- 질문형 문장(‘왜’, ‘어떻게’)을 사용하지 않는다.
- **중복된 상품·서비스 용어** 혹은 **중복된 성능·품질 용어**를 포함한 문장은 출력하지 않는다.  
- **VOC 고객경험요소**에 해당하지 않는 문장은 제외한다.

---

## 6️⃣ 최종 지시  

당신은 위 규칙을 **절대적으로** 따르며, **모든** 입력 VOC에 대해 원인‑결과 형태의 문제원인 인사이트 문장을 생성한다.  
출력 형식은 **텍스트만**이며, 각 문장은 **개행**(`\n`)으로 구분한다.
""".strip()

In [33]:
"""
프롬프트 v0 (No CXE)
"""
system_prompt_no_cxe_v0 = """
# 은행 NPS 설문조사 VOC 문제원인 식별 전문가 에이전트  

## 1️⃣ 역할  
당신은 고객 VOC(Voice of Customer) 텍스트를 분석하여, 문제원인을 요약한 문장으로 변환하는 문제원인 식별 전문가입니다.  
목표는 **고객이 직접 언급한 핵심 대상(무엇)**과 **그 대상에 대해 경험·평가한 특성·행위(어떻게)**만을 남겨, 의사결정에 바로 활용할 수 있는 문제 원인에 대한 **간결한 인사이트**를 제공하는 것입니다.  

---

## 2️⃣ 입력값 정의  

| 입력 항목 | 설명 | 예시 |
|----------|------|------|
| **VOC 설문문항채널** | 고객이 의견을 제시한 접점·채널 | “모바일 앱”, “콜센터”, “오프라인 매장” |
| **VOC 고객경험단계 리스트** | VOC 설문문항채널에서 발생할 수 있는 고객경험단계의 리스트 | ['내점/방문', '대기', '맞이/의도파악', '상담', '업무처리/배웅'] |
| `문항 질문` (Nullable) | 설문에서 묻는 구체적인 질문 (객관식) | "Q5-1.친구 또는 지인에게 상담한 직원과의 고객맞이/방문목적파악 경험을 적극적으로 추천하는 이유는 무엇입니까?" |
| `문항 선택 항목` (Nullable) | '문항 질문'에 대한 고객이 선택한 답변이다. | "친절한 화법과 적절한 호칭 사용" |
| `VOC 원문` | 하나 이상의 문장(또는 의미 단위)으로 구성된 고객의 서술형 의견 | `친절하고 상냥해서 좋았다.` |
| `VOC 상품·서비스 용어` (Nullable) | `VOC 원문`으로 부터 추출된 상품·서비스 용어 | "로그인", "앱화면" |
| `VOC 성능·품질 용어` (Nullable) | `VOC 원문`으로 부터 추출된 성능·품질 용어 | "불편하다", "복잡하다" |


---

## 3️⃣ 용어 정의  

| 구분 | 정의 | 주요 품사 | 특징 |
|------|------|-----------|------|
| **상품·서비스 용어** | 고객이 직접 인지·언급하는 **‘무엇(대상)’**. 구체적인 사물·서비스·수량·순서를 가리킴. | 명사 | 1️⃣ 구체적 사물·서비스 (예: “스마트폰”, “배송”, “프리미엄 플랜”) |
| **성능·품질 용어** | 고객이 **‘어떻게(특성·행위)’** 경험·평가한 **‘특성·행위·상태’**. 상태·속성·성과를 서술한다. | 형용사, 동사, 형용사형·관형사형 전성어미 등 용언 전반 | 1️⃣ 상태·속성·성과 서술 (예: “빠르다”, “불편하다”) <br>2️⃣ 정량·정성 평가 모두 포함 (예: “높은 만족도”, “오류가 적다”) <br>3️⃣ 비교·변화 표현 가능 (예: “전보다 개선되다”, “가장 저렴한”) |
| **의미 단위** | 문장 구분이 애매할 경우, “쉼표·‘그리고’·‘하지만’·‘그러나’” 등으로 구분한 **독립적인 의견 조각** |

---

## 4️⃣ 문제원인 식별 규칙 (절대 준수)

1. **입력된 VOC 분류 결과들 파악하기**  
    - 제공된 **VOC 설문문항채널**과 해당하는 **VOC 고객경험단계 리스트**를 파악하여 VOC에서 어떤 주제를 강조해야하는지 이해한다.
    - **VOC 상품·서비스 용어**와 **VOC 성능·품질 용어**를 생성할 <원인>, <결과>에 참고한다.

2. **연관 문장·의미 찾기**  
    - VOC 원문 전체를 스캔하여, **VOC 설문문항채널**, **VOC 고객경험단계 리스트**와 **VOC 상품·서비스 용어**, **VOC 성능·품질 용어**에 가장 직접적으로 연관되는 문장(또는 의미 단위)를 식별한다.
    - **고객 부정적 감정이 가장 강하게 드러나는** 하나만 남긴다.
        -> **강조어·부정적 감정의 정도·빈도**를 기준으로 우선순위 결정.
    - **반드시 연관된 단위들만**을 식별한다. 

3. **문제원인 인사이트 생성**
    - 식별된 연관 단위에서 문제원인 인사이트를 생성한다.
    - 형식: **`<원인> <결과>.`**
    - ‘가/이’ 등등의 조사는 대상에 맞게 자동 선택한다.
    - 불필요한 부사·보조어는 제거한다.
    - "~임", "~음", "~함", "~로 보여짐", "~나타남" 등 개조식문체와 종경어미로 생성한다.


4. **예외 처리**
    - 문제원인이 존재하지 않으면 "채널, 고객경험단계의 문제원인이 식별되지 않음"을 응답한다.

---

## 5️⃣ 금지 사항  

- **원문 그대로 복사**하거나 **불필요한 부연 설명**을 포함하지 않는다.
- 원문이 '없다', '내용무' 등 의미 없는 경우 요약하지 않는다.
- 질문형 문장(‘왜’, ‘어떻게’)을 사용하지 않는다.
- **중복된 상품·서비스 용어** 혹은 **중복된 성능·품질 용어**를 포함한 문장은 출력하지 않는다.  
- **VOC 설문문항채널**에 해당하지 않는 문장은 제외한다.
- **VOC 고객경험단계 리스트**에 해당하지 않는 문장은 제외한다.

---

## 6️⃣ 최종 지시  

당신은 위 규칙을 **절대적으로** 따르며, **모든** 입력 VOC에 대해 원인‑결과 형태의 문제원인 인사이트 문장을 생성한다.  
출력 형식은 **텍스트만**이며, 각 문장은 **개행**(`\n`)으로 구분한다.
""".strip()

In [28]:
"""
인입 VOC
"""
ch = '영업점'
cx_stge_list = ['내점/방문','대기','맞이/의도파악','직원상담','업무처리/배웅']


# 추천이유 응답 내용 보다 VOC에서 추출
voc1 = {
    'content': "대기 시간이 길어요",
    'question': "Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?",
    'answer': '확인하기 쉬운 대기정보(대기순번, 대기예상시간)',
    'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |',
    'ch': ch,
    'stge': cx_stge_list[1],
    'p_s': '대기시간',
    'p_q': '길었다',
}

voc2 = {
    'content': "ATM기 이용이 정말정말 불편해요!!!",
    'question': "Q2-1.친구 또는 지인에게 ${부서명}지점의 영업점 내점/방문과정 경험을 비추천하는 이유는 무엇입니까?",
    'answer': '이용하기 불편한 자동화기기(ATM, STM 등)',
    'cxe': '| ATM 이용 안내의 명확성 | 장애인/외국인 등 약자를 위한 음성 안내, 다국어 지원 기능의 유무 및 품질 |',
    'ch': ch,
    'stge': cx_stge_list[0],
    'p_s': 'ATM 이용',
    'p_q': '편하다',
}

# 고객경험요소에 따라
# 고객경험단계에 따라
voc3 = {
    'content': "상담을통한 부적절한 상품과다양한상품에대한 불충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |',
    'ch': ch,
    'stge': cx_stge_list[3],
    'p_s': '상품설명',
    'p_q': '불충분',
}
voc4 = {
    'content': "상담을통한 부적절한 상품과다양한상품에대한 불충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'cxe': '| 신속한 상담 및 업무 처리 | 상담 시간과 업무 처리 시간을 효율적으로 관리하여 고객 대기 시간 최소화 |',
    'ch': ch,
    'stge': cx_stge_list[4],
    'p_s': '고객',
    'p_q': '답답',
}

# 개체어에 따라
voc5 = {
    'content': "상담을통한 부적절한 상품제안과 상품 다양성에대한 불충분한 설명을 듣는다\n객장은 속도가 빠르고 고객의입장에서답답하다. 업무가 빨라서 기대된다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |',
    'ch': '플랫폼',
    'stge': '로그인/인증',
    'p_s': '',
    'p_q': '',
}
voc6 = {
    'content': "상담을통한 부적절한 상품제안과 상품 다양성에대한 불충분한 설명을 듣는다\n객장은 속도가 빠르고 고객의입장에서답답하다. 업무가 빨라서 기대된다",
    'question': "개선의견 (주관식)",
    'answer': '',
    'ch': ch,
    'stge': cx_stge_list,
    'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |',
    'p_s': '상품 다양성',
    'p_q': '불충분',
}

vocs = [ voc1, voc2, voc3, voc4, voc5, voc6 ]
vocs2 = [ voc1, voc2, voc3, voc4, voc5 ]

In [20]:
"""
입력 세팅 및 결과 생성 V0 - CXE
"""
from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs:

    print(f"Problem Reason Generate Index: {index} VOC:")
    print(f"{voc}")
    index = index + 1
    
    human_prompt = f"""
    ## 입력:
    
    **문항 질문**
    {voc['question']}
    
    ---
    
    **문항 응답내용**
    {voc['answer']}
    
    ---
    
    **VOC 원문**
    {voc['content']}
    
    ---

    **VOC 상품·서비스 용어**
    {voc['p_s']}

    ---
    
    **VOC 성능·품질 용어**
    {voc['p_q']}

    ---
    
    **VOC 고객경험요소**
    | 고객경험요소 | 설명 |
    |--------------|------|
    {voc['cxe']}
    """.strip()

    messages = [
        SystemMessage(content=system_prompt_cxe_v0),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")
    

Problem Reason Generate Index: 0 VOC:
{'content': '대기 시간이 길어요', 'question': 'Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?', 'answer': '확인하기 쉬운 대기정보(대기순번, 대기예상시간)', 'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |', 'p_s': '대기시간', 'p_q': '길었다'}
대기시간이 길어 대기경험 만족도가 저하됨.
Problem Reason Generate Index: 1 VOC:
{'content': 'ATM기 이용이 정말정말 불편해요!!!', 'question': 'Q2-1.친구 또는 지인에게 ${부서명}지점의 영업점 내점/방문과정 경험을 비추천하는 이유는 무엇입니까?', 'answer': '이용하기 불편한 자동화기기(ATM, STM 등)', 'cxe': '| ATM 이용 안내의 명확성 | 장애인/외국인 등 약자를 위한 음성 안내, 다국어 지원 기능의 유무 및 품질 |', 'p_s': 'ATM 이용', 'p_q': '편하다'}
ATM 이용의 불편함이 영업점 방문 과정 비추천 요인임.
Problem Reason Generate Index: 2 VOC:
{'content': '상담을통한 부적절한 상품과다양한상품에대한 불충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다', 'question': '개선의견 (주관식)', 'answer': '', 'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |', 'p_s': '상품설명', 'p_q': '불충분'}
상품설명 불충분이 부적절한 상품 제시를 초래함.
Problem Reason Generate Index: 3 VOC:
{'content': '상담

In [34]:
"""
입력 세팅 및 결과 생성 V0 - Noe CXE
"""
from langchain.schema import BaseMessage, HumanMessage, SystemMessage

index = 0

for voc in vocs2:

    print(f"Problem Reason Generate Index: {index} VOC:")
    print(f"{voc}")
    index = index + 1
    
    human_prompt = f"""
    ## 입력:

    **VOC 설문문항채널**
    {voc['ch']}
    
    ---

    **VOC 고객경험단계 리스트**
    {voc['stge']}
    
    ---
    
    **문항 질문**
    {voc['question']}
    
    ---
    
    **문항 응답내용**
    {voc['answer']}
    
    ---
    
    **VOC 원문**
    {voc['content']}
    
    ---

    **VOC 상품·서비스 용어**
    {voc['p_s']}

    ---
    
    **VOC 성능·품질 용어**
    {voc['p_q']}
    """.strip()

    messages = [
        SystemMessage(content=system_prompt_no_cxe_v0),
        HumanMessage(content=human_prompt)
    ]

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
    print("\n==================================================")
    

Problem Reason Generate Index: 0 VOC:
{'content': '대기 시간이 길어요', 'question': 'Q3-1.친구 또는 지인에게 ${부서명}지점의 상담 전 대기경험을 적극적으로 추천하는 이유는 무엇입니까?', 'answer': '확인하기 쉬운 대기정보(대기순번, 대기예상시간)', 'cxe': '| 대기 고객의 집중도 분산 | 대기 고객이 지루함을 잊을 수 있도록 정보성/흥미성 콘텐츠 제공 |', 'ch': '영업점', 'stge': '대기', 'p_s': '대기시간', 'p_q': '길었다'}
영업점 상담 전 대기시간이 길다.
Problem Reason Generate Index: 1 VOC:
{'content': 'ATM기 이용이 정말정말 불편해요!!!', 'question': 'Q2-1.친구 또는 지인에게 ${부서명}지점의 영업점 내점/방문과정 경험을 비추천하는 이유는 무엇입니까?', 'answer': '이용하기 불편한 자동화기기(ATM, STM 등)', 'cxe': '| ATM 이용 안내의 명확성 | 장애인/외국인 등 약자를 위한 음성 안내, 다국어 지원 기능의 유무 및 품질 |', 'ch': '영업점', 'stge': '내점/방문', 'p_s': 'ATM 이용', 'p_q': '편하다'}
ATM 이용이 불편하다.
Problem Reason Generate Index: 2 VOC:
{'content': '상담을통한 부적절한 상품과다양한상품에대한 불충분한설명을 듣는다\n객장은 속도가느리고 고객의입장에서답답하다. 좀더빠른업무기대한다\n상담실을통한 은행일을본다 객장에서볼일은 가급적하지않으려고한다', 'question': '개선의견 (주관식)', 'answer': '', 'cxe': '| 다양한 상품 비교 및 제시 | 고객에게 2~3개 이상의 대안 상품을 비교하며 장단점을 명확히 설명 |', 'ch': '영업점', 'stge': '직원상담', 'p_s': '상품설명', 'p_q': '불충분'}
직원상담의 상품설명이 불충분하다